# Verify the 6 MS(1190) stragglers under the sum cap (L=100) @ 1M nodes

The plain-`n=2` GS-Sub baseline solves **634** of the 640 "solvable" presentations under the fair
`per_relator L=24` cap. Six of them — **idx 634–639** — need the paper's looser **sum cap**
(`|r1| + |r2| < 100`) to reach trivial. This notebook reproduces the paper's **640 @ 1M** by running
exactly those six at a **1,000,000-node** budget under the sum cap, and replays every solve through the
solver's own `verify_path`.

**Why Colab, not a laptop.** Each 1M sum-cap run peaks at **~24 GB** RAM (measured: 4.9 GB @ 200k nodes,
growth ~linear — the visited dict keeps one string-tuple key per *pushed* neighbor, and under the sum cap
relators grow to ~100 so almost nothing is capped out). A 16 GB machine gets jetsam-killed around 600k
nodes. **You must use a High-RAM runtime:** Runtime → Change runtime type → **High-RAM** (~51 GB, Colab
Pro). Free Colab (~13 GB) is *not* enough — on free Colab drop `BUDGET` to ~400k for a partial run.

**Time / resumability.** ≈1 h per idx (≈305 nodes/s measured) ⇒ ~5–6 h for all six at `WORKERS=1`. The
stream is **append-only, resumable** (JSONL on Drive), so a preempted session just continues where it
stopped — re-run all cells and only the unfinished idx run.

This is a *thin wrapper* over the repo's verbatim solver (`experiments/stable_ac/baseline_n2/greedy_ac.py`);
no solver logic is redefined here.

In [ ]:
# --- parameters -------------------------------------------------------------
REPO_URL  = "https://github.com/Avi161/ACSolverX.git"
BRANCH    = "test/stable-ac-moves"
DRIVE_OUT = "/content/drive/MyDrive/acsolverx_results"   # stream persists here across preemption
USE_DRIVE = True             # False -> write beside the repo instead

IDXS      = [634, 635, 636, 637, 638, 639]   # the 6 stragglers (paper-solvable, unsolved under L=24)
BUDGET    = 1_000_000        # nodes  (the paper's 1M tier)
MAX_LEN   = 100              # sum cap:  |r1| + |r2| < MAX_LEN   ("L=100")
CAP       = "sum"            # 'sum' reproduces paper Table 1 (634@100k / 640@1M)
WORKERS   = 1               # each run peaks ~24 GB; 1 fits a 51 GB High-RAM box (2 is tight)
OUT_NAME  = "stragglers_sum_L100_1M.jsonl"

In [ ]:
# --- setup: mount Drive, clone repo, install deps ---------------------------
import os, sys, subprocess, json

IN_COLAB = "google.colab" in sys.modules
if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_OUT, exist_ok=True)

# locate or clone the repo
if os.path.isdir("ACSolverX/.git"):
    REPO = os.path.abspath("ACSolverX")
elif os.path.exists("experiments/stable_ac/baseline_n2/greedy_ac.py"):
    REPO = os.getcwd()                          # already inside the repo
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, "ACSolverX"], check=True)
    REPO = os.path.abspath("ACSolverX")
print("repo:", REPO)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numba", "numpy"], check=True)
sys.path.insert(0, os.path.join(REPO, "experiments/stable_ac/baseline_n2"))

In [ ]:
# --- import the verbatim solver + load the 6 presentations ------------------
from greedy_ac import solve_one           # thin wrapper: runs ACRelatorSolver + verify_path

labels = json.load(open(os.path.join(REPO, "results/labels_1190.json")))
PRES = {r["idx"]: r["presentation"] for r in labels}    # flat signed-int, len 48
for i in IDXS:
    assert i in PRES, f"idx {i} missing from labels_1190.json"

OUT_DIR = DRIVE_OUT if (USE_DRIVE and IN_COLAB) else os.path.join(REPO, "results/baseline_greedy/stragglers")
os.makedirs(OUT_DIR, exist_ok=True)
OUT = os.path.join(OUT_DIR, OUT_NAME)
print("loaded idx:", IDXS)
print("stream ->", OUT)

In [ ]:
# --- base-case gate: idx 0 must solve + verify (also warms numba pre-fork) --
bc = solve_one(PRES[0], cap_mode=CAP, max_len=MAX_LEN, max_nodes=100_000)
print("base case idx 0:", bc)
assert bc["solved"] and bc["path_verified"], "base case FAILED - do not trust the sweep"
print("OK base case passed; numba is warm (forked children inherit the compiled cache)")

In [ ]:
# --- run the 6 @ 1M: resumable, fresh forked child per idx (peak RSS released) --
import time, resource, multiprocessing as mp

def jsonl_done(path):
    done = set()
    if os.path.exists(path):
        for line in open(path):
            line = line.strip()
            if not line:
                continue
            try:
                done.add(json.loads(line)["idx"])
            except Exception:
                pass                             # ignore a torn trailing line; that idx re-runs
    return done

def worker(idx):
    from greedy_ac import solve_one              # re-import inside the forked child
    r = solve_one(PRES[idx], cap_mode=CAP, max_len=MAX_LEN, max_nodes=BUDGET)
    # Linux (Colab): ru_maxrss is in KB
    r.update(idx=idx, budget_nodes=BUDGET, max_len=MAX_LEN, cap=CAP,
             peak_rss_gb=round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024 / 1024, 2))
    return r

done = jsonl_done(OUT)
todo = [i for i in IDXS if i not in done]
print(f"{len(done)} already done {sorted(done)};  running {todo}")

ctx = mp.get_context("fork")                     # Colab is Linux/fork -> a notebook-defined worker is safe
with open(OUT, "a") as f:
    with ctx.Pool(processes=WORKERS, maxtasksperchild=1) as pool:   # fresh child per idx
        for r in pool.imap_unordered(worker, todo):
            f.write(json.dumps(r) + "\n"); f.flush(); os.fsync(f.fileno())
            tag = "*** SOLVED ***" if r["solved"] else "unsolved"
            print(f"  idx {r['idx']}: {tag}  nodes={r['nodes_explored']:,}  "
                  f"verified={r['path_verified']}  path_len={r['path_len']}  "
                  f"peak={r['peak_rss_gb']} GB  {r['wall_time_s']/60:.1f} min")
print("done ->", OUT)

In [ ]:
# --- summary ---------------------------------------------------------------
rows = {json.loads(l)["idx"]: json.loads(l) for l in open(OUT) if l.strip()}
solved   = [i for i in IDXS if rows.get(i, {}).get("solved") and rows[i].get("path_verified")]
unsolved = [i for i in IDXS if i not in solved]

print(f"stragglers solved AND verified @ {BUDGET:,} nodes (sum cap L={MAX_LEN}):  {len(solved)}/6  {solved}")
print(f"still unsolved:  {unsolved}")
print(f"=> total MS(1190) solved = 634 (idx 0-633, L=24) + {len(solved)} here = "
      f"{634 + len(solved)}     (paper Table 1: 640 @ 1M)")
print()
for i in IDXS:
    r = rows.get(i, {})
    print(f"  idx {i}: solved={r.get('solved')} verified={r.get('path_verified')} "
          f"nodes={r.get('nodes_explored')} path_len={r.get('path_len')} "
          f"max_len_along_path={r.get('max_len_along_path')} peak={r.get('peak_rss_gb')}GB")